# Train and save an ML-dSGP4 model

This tutorial shows the mechanics needed to train `dsgp4.mldsgp4`, save its weights, and load them again.

ML-dSGP4 is intended to learn corrections when higher-precision simulated or observed TEME states are available. To keep this example self-contained, the target below is **synthetic**: it applies a small deterministic scale correction to ordinary dSGP4 states. It is only a teaching target, not a replacement for a high-precision propagator or observations.


## Imports and reproducibility

dSGP4 uses `torch.float64` by default, which is also appropriate for the state vectors used here.


In [ ]:
import dsgp4
import torch

torch.manual_seed(0)


## Build a small training set

Each training item consists of a TLE, a time since the TLE epoch in minutes, and a reference TEME state `[x, y, z, vx, vy, vz]`.

For a real application, replace `reference_states_teme` with states from your higher-precision propagator or observations, aligned to the same TLEs and epochs.


In [ ]:
source_tle = dsgp4.tle.load("example.tle")[0]

tsinces = torch.linspace(0.0, 6.0 * 60.0, 64)
train_tsinces = tsinces[:48]
validation_tsinces = tsinces[48:]

train_tles = [source_tle] * len(train_tsinces)
validation_tles = [source_tle] * len(validation_tsinces)

with torch.no_grad():
    train_reference_states_teme = dsgp4.propagate_batch(
        train_tles, train_tsinces, initialized=False
    ).reshape(-1, 6)
    validation_reference_states_teme = dsgp4.propagate_batch(
        validation_tles, validation_tsinces, initialized=False
    ).reshape(-1, 6)


## Normalize the target states

`mldsgp4.forward()` returns normalized position and velocity. Targets used in the loss must therefore use the model's `normalization_R` and `normalization_V`.

The small scale vector below only makes the example trainable without an external truth dataset.


In [ ]:
model_kwargs = {
    "hidden_size": 35,
    "normalization_R": 6958.137,
    "normalization_V": 7.947155867983262,
}
model = dsgp4.mldsgp4(**model_kwargs)

def normalize_states(states_teme):
    return torch.cat(
        (
            states_teme[:, :3] / model.normalization_R,
            states_teme[:, 3:] / model.normalization_V,
        ),
        dim=1,
    )

synthetic_scale = torch.tensor(
    [1.00020, 0.99980, 1.00010, 1.00030, 0.99970, 1.00020]
)

train_targets = normalize_states(train_reference_states_teme) * synthetic_scale
validation_targets = (
    normalize_states(validation_reference_states_teme) * synthetic_scale
)


## Train with ordinary PyTorch

Because dSGP4 is differentiable, standard PyTorch optimizers can backpropagate through the propagation step. The example uses mean-squared error in normalized state space.

For real datasets, tune the optimizer, learning rate, batching strategy, and loss weighting to the precision and scale of your reference data.


In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_function = torch.nn.MSELoss()

for epoch in range(100):
    model.train()
    optimizer.zero_grad()

    prediction = model(train_tles, train_tsinces)
    loss = loss_function(prediction, train_targets)
    loss.backward()
    optimizer.step()

    if epoch % 20 == 0 or epoch == 99:
        model.eval()
        with torch.no_grad():
            validation_prediction = model(
                validation_tles, validation_tsinces
            )
            validation_loss = loss_function(
                validation_prediction, validation_targets
            )
        print(
            f"epoch={epoch:03d} "
            f"train_loss={loss.item():.3e} "
            f"validation_loss={validation_loss.item():.3e}"
        )


## Save and load the trained weights

`save_model()` stores the PyTorch state dictionary. Recreate the model with the same architecture and normalization arguments before loading the weights; those constructor settings are not stored inside the state dictionary.


In [ ]:
model_path = "mldsgp4_trained_example.pth"
model.save_model(model_path)

restored_model = dsgp4.mldsgp4(**model_kwargs)
restored_model.load_model(model_path, device="cpu")

for name, parameter in model.state_dict().items():
    torch.testing.assert_close(
        parameter, restored_model.state_dict()[name]
    )


## Using real high-precision or observed data

For a production training set:

1. Associate every reference state with the TLE and `tsince` used to predict it.
2. Express reference position and velocity in the TEME frame expected by this model, in km and km/s before normalization.
3. Normalize targets with the same `normalization_R` and `normalization_V` supplied to `mldsgp4`.
4. Split training and validation data by satellite and/or time in a way that measures the generalization you care about, rather than randomly leaking neighboring samples between splits.
5. Record the constructor arguments together with the saved state dictionary so the model can be reconstructed correctly later.

The pretrained weights shipped with the inference tutorial are an example model; this training workflow is the path to producing weights for a different satellite population or reference dataset.
